In [1]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict, Audio
import soundfile as sf
from pathlib import Path
from huggingface_hub import HfApi, create_repo
import librosa
import uuid


audio_dir = Path("dataset_backup/audio")
test_audio_dir = Path("dataset_backup/test_audio")
transcription_file = Path("dataset_backup/transcriptions.csv")
test_transcription_file = Path("dataset_backup/test_transcriptions.csv")

def load_data(audio_dir, transcription_file):
    df = pd.read_csv(transcription_file)
    audio_files = [str(file) for file in audio_dir.glob("*.wav")]
    audio_data = []

    for audio_file in audio_files:
        audio_data.append({
            "audio": audio_file,  
            "transcription": df[df["audio_file"] == os.path.basename(audio_file)]["transcription"].values[0]
        })

    dataset = Dataset.from_list(audio_data)
    dataset = dataset.cast_column("audio", Audio())  
    return dataset


train_dataset = load_data(audio_dir, transcription_file)
test_dataset = load_data(test_audio_dir, test_transcription_file)

new_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})


print(new_dataset)


DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 507
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 159
    })
})


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [3]:
from huggingface_hub import login, HfApi
import datasets


login(token=hf_token)

repo_name = "ufcg-labmet-fala-texto-main-final"

api = HfApi()
api.create_repo(repo_id=repo_name, private=False)

output_dir = "./huggingface_datasets"
new_dataset.save_to_disk(output_dir)

dataset = datasets.load_from_disk(output_dir)
dataset.push_to_hub(repo_name, private=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Saving the dataset (0/1 shards):   0%|          | 0/507 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/159 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/159 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/brunopbb/ufcg-labmet-fala-texto-main-final/commit/9c7cc30562417cad50379bce0880ca15a3b8de07', commit_message='Upload dataset', commit_description='', oid='9c7cc30562417cad50379bce0880ca15a3b8de07', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/brunopbb/ufcg-labmet-fala-texto-main-final', endpoint='https://huggingface.co', repo_type='dataset', repo_id='brunopbb/ufcg-labmet-fala-texto-main-final'), pr_revision=None, pr_num=None)